# PKG Attrition — Schema Delta Diagnostics

*Companion to `pkg_attrition_source_eda_v4`. Profiles only the columns that
were not available when v4 was built: `acct_status`, `closed_dt`,
`deposit_family`, `avg_monthly_bal_1`, `cpty_name`, `category`.*

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

Read-only profiling of the Neo4j payments staging table and the deposit panel,
ahead of building the deposit-anchored working tables.

Nothing here writes to disk or to Hive. Every section aggregates in Spark and
brings only small result frames to pandas.

**What this notebook decides**

| § | Question | Consumer |
|---|---|---|
| 1.2 | Which `deposit_family` values are in scope | scoping decision, needs your call |
| 1.3 | Does an account keep appearing after `closed_dt` | survivorship; caps observable departures |
| 1.4 | Is `avg_monthly_bal_1` month-to-date or prior-complete-month | balance series construction |
| 1.6 | Closure / empty definitions and how they agree | the departure label |
| 2.1 | Null-pattern topology of the id columns | direction logic |
| 2.2 | Does one economic payment occupy one row or two | every dollar total in the study |
| 2.5 | `cpty_name` coverage and exact-match rate against `customer_name` | `same_name_outflow_flag` |
| 3.x | Account join coverage, `mdm_id` ↔ `cust_pwr_id` cardinality | the panel population |

Standing rules applied: ids are strings end to end; no `ntile` over a global
window; no `groupby.apply`; raw examples are `LIMIT`-ed, never collected whole.

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, Window

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# ----------------------------------------------------------------------------
# CONFIG — fill these in
# ----------------------------------------------------------------------------
PAY_TBL = "bdahd01p_dlcdi1_cdi_tm.neo4j_payments"   # staging transactions
DEP_TBL = "bdahd01p_dlcdi1_cdi_tm.<deposit_table>"  # deposit panel

START = "2024-01-01"
END   = "2026-07-31"

# Sampling knobs for the expensive per-account tests. Raise once the cheap
# sections have run and the table sizes are known.
SEM_SAMPLE_ACCTS = 20_000   # §1.4 avg_monthly_bal_1 semantics
DUP_SAMPLE_DAYS  = 14       # §2.2 duplication test window

# Column names, isolated here so a rename upstream is a one-line fix.
C = dict(
    # payments
    txn_id="transaction_id", txn_dt="trans_dt", amt="amount",
    rail="payment_rail", cat="category",
    mdm_p="mdm_id_pays", mdm_r="mdm_id_receives",
    nm_p="customer_name_pays", nm_r="customer_name_receives",
    acct_p="pnc_dep_acct_pays", acct_r="pnc_dep_acct_receives",
    cpty_id="unq_cpty_acct_id", cpty_nm="cpty_name", cpty_fi="cpty_fin_entity_name",
    # deposits
    d_acct="acct_full_acct_id", d_load="edw_tda_load_dt", d_bal="balance",
    d_avg1="avg_monthly_bal_1", d_status="acct_status", d_fam="deposit_family",
    d_cust="cust_pwr_id", d_name="cust_name", d_open="opened_dt", d_close="closed_dt",
)


def q(sql, n=None):
    """Run SQL, return pandas. `n` guards against pulling something large."""
    df = spark.sql(sql)                                     # noqa: F821
    pdf = df.limit(n).toPandas() if n else df.toPandas()
    return pdf


def show(title, pdf):
    print(f"\n{'=' * 78}\n{title}\n{'=' * 78}")
    print(pdf.to_string(index=False) if len(pdf) else "(empty)")
    return pdf

## 0. Column inventory

Run this first. It answers what the schema actually holds rather than what we
think it holds: whether `avg_monthly_bal_2` / `_3` exist, what `deposit_family`
and `category` are typed as, and whether either table is partitioned.

In [ ]:
show("PAYMENTS — schema", q(f"DESCRIBE {PAY_TBL}"))
show("DEPOSITS — schema", q(f"DESCRIBE {DEP_TBL}"))

for tbl in (PAY_TBL, DEP_TBL):
    try:
        parts = q(f"SHOW PARTITIONS {tbl}")
        print(f"\n{tbl}: {len(parts)} partitions, e.g. {parts.head(3).values.tolist()}")
    except Exception as e:
        print(f"\n{tbl}: not partitioned or SHOW PARTITIONS unavailable ({type(e).__name__})")

In [ ]:
# Base views, date-filtered once. Everything downstream reads these.
spark.sql(f"""                                                     -- noqa: F821
CREATE OR REPLACE TEMPORARY VIEW pay AS
SELECT * FROM {PAY_TBL}
WHERE  {C['txn_dt']} >= '{START}' AND {C['txn_dt']} <= '{END}'
""")

spark.sql(f"""                                                     -- noqa: F821
CREATE OR REPLACE TEMPORARY VIEW dep AS
SELECT * FROM {DEP_TBL}
WHERE  {C['d_load']} >= '{START}' AND {C['d_load']} <= '{END}'
""")

show("Row counts in window", q("""
SELECT 'payments' AS tbl, COUNT(*) AS rows FROM pay
UNION ALL
SELECT 'deposits', COUNT(*) FROM dep
"""))

---
## 1. Deposit panel

### 1.1 Load-date calendar

One row per account per business day is the stated shape. This tests it:
missing dates, days with anomalous row counts, and whether the account count
drifts (attrition) or steps (a reload or a scope change).

In [ ]:
cal = show("Load dates — rows and accounts per day", q(f"""
SELECT {C['d_load']}                        AS load_dt,
       COUNT(*)                             AS rows,
       COUNT(DISTINCT {C['d_acct']})        AS accts,
       SUM(CASE WHEN {C['d_status']} = 'C' THEN 1 ELSE 0 END) AS status_c_rows
FROM   dep
GROUP  BY {C['d_load']}
ORDER  BY 1
"""))

cal["load_dt"] = pd.to_datetime(cal["load_dt"])
bdays = pd.bdate_range(cal["load_dt"].min(), cal["load_dt"].max())
missing = bdays.difference(cal["load_dt"])
print(f"\nBusiness days in range : {len(bdays)}")
print(f"Load dates present     : {len(cal)}")
print(f"Business days missing  : {len(missing)}")
if len(missing):
    print("First 40 missing:", [d.date().isoformat() for d in missing[:40]])

# Rows-per-account should be ~1.0 on every day. Anything above it is duplication
# in the panel itself, which would corrupt every balance series.
cal["rows_per_acct"] = cal["rows"] / cal["accts"]
print(f"\nrows_per_acct  min={cal.rows_per_acct.min():.4f}  "
      f"median={cal.rows_per_acct.median():.4f}  max={cal.rows_per_acct.max():.4f}")
show("Days where rows_per_acct > 1.001",
     cal.loc[cal.rows_per_acct > 1.001, ["load_dt", "rows", "accts", "rows_per_acct"]])

### 1.2 `deposit_family` profile — the scoping decision

The brief scopes to the corporate DDA / MMDA book. The panel holds everything.
This table is what that scoping decision should be made from: account counts,
balance mass, and how much payment activity each family actually carries.

In [ ]:
show("deposit_family — account and balance profile", q(f"""
WITH latest AS (
  SELECT MAX({C['d_load']}) AS mx FROM dep
)
SELECT d.{C['d_fam']}                                   AS deposit_family,
       COUNT(DISTINCT d.{C['d_acct']})                  AS accts_on_last_day,
       COUNT(DISTINCT d.{C['d_cust']})                  AS customers,
       ROUND(SUM(d.{C['d_bal']}) / 1e6, 1)              AS balance_mm,
       ROUND(AVG(d.{C['d_bal']}), 0)                    AS mean_bal,
       ROUND(PERCENTILE_APPROX(d.{C['d_bal']}, 0.5), 0) AS median_bal,
       SUM(CASE WHEN d.{C['d_bal']} < 0 THEN 1 ELSE 0 END) AS negative_bal_rows
FROM   dep d CROSS JOIN latest l
WHERE  d.{C['d_load']} = l.mx
GROUP  BY d.{C['d_fam']}
ORDER  BY balance_mm DESC
"""))

show("deposit_family — accounts ever seen, and closures in window", q(f"""
SELECT {C['d_fam']}                                              AS deposit_family,
       COUNT(DISTINCT {C['d_acct']})                             AS accts_ever,
       COUNT(DISTINCT CASE WHEN {C['d_close']} IS NOT NULL
                           THEN {C['d_acct']} END)               AS accts_with_closed_dt,
       COUNT(DISTINCT CASE WHEN {C['d_status']} = 'C'
                           THEN {C['d_acct']} END)               AS accts_ever_status_c
FROM   dep
GROUP  BY {C['d_fam']}
ORDER  BY accts_ever DESC
"""))

### 1.3 Closed-account persistence — the survivorship question

Three things to separate:

1. Does an account keep producing rows after its `closed_dt`?
2. Do accounts vanish from the panel without ever carrying a `closed_dt`?
3. Does the panel contain any account that closed *before* the window starts?

(3) is the one that caps the study. If the panel is a series of current-state
snapshots appended forward, then no pre-window closure is in it at all, and the
observable departure history is the panel window, not the account history.

In [ ]:
acct_span = spark.sql(f"""                                          -- noqa: F821
SELECT {C['d_acct']}                       AS acct,
       MIN({C['d_load']})                  AS first_seen,
       MAX({C['d_load']})                  AS last_seen,
       COUNT(*)                            AS n_days,
       MAX({C['d_open']})                  AS opened_dt,
       MAX({C['d_close']})                 AS closed_dt,
       MAX(CASE WHEN {C['d_status']} = 'C' THEN 1 ELSE 0 END) AS ever_status_c,
       MIN(CASE WHEN {C['d_status']} = 'C' THEN {C['d_load']} END) AS first_status_c_dt
FROM   dep
GROUP  BY {C['d_acct']}
""")
acct_span.createOrReplaceTempView("acct_span")
acct_span.cache().count()

show("Accounts by closure evidence", q("""
SELECT closed_dt IS NOT NULL AS has_closed_dt,
       ever_status_c          AS ever_status_c,
       COUNT(*)               AS accts
FROM   acct_span
GROUP  BY 1, 2
ORDER  BY 1, 2
"""))

# Q1: rows after closed_dt
show("Days the account survives past closed_dt (deciles)", q("""
SELECT PERCENTILE_APPROX(DATEDIFF(last_seen, closed_dt), ARRAY(0.0,0.1,0.25,0.5,0.75,0.9,1.0))
         AS days_last_seen_minus_closed,
       COUNT(*) AS accts,
       SUM(CASE WHEN DATEDIFF(last_seen, closed_dt) >  0 THEN 1 ELSE 0 END) AS survives_past_close,
       SUM(CASE WHEN DATEDIFF(last_seen, closed_dt) =  0 THEN 1 ELSE 0 END) AS ends_on_close,
       SUM(CASE WHEN DATEDIFF(last_seen, closed_dt) <  0 THEN 1 ELSE 0 END) AS ends_before_close
FROM   acct_span
WHERE  closed_dt IS NOT NULL
"""))

# What does a post-closure row look like? Balance zero, or still moving?
show("Balance on rows dated after closed_dt", q(f"""
SELECT CASE WHEN DATEDIFF(d.{C['d_load']}, s.closed_dt) BETWEEN 1 AND 7   THEN '1-7d'
            WHEN DATEDIFF(d.{C['d_load']}, s.closed_dt) BETWEEN 8 AND 30  THEN '8-30d'
            WHEN DATEDIFF(d.{C['d_load']}, s.closed_dt) > 30              THEN '30d+'
       END                                                AS bucket,
       COUNT(*)                                           AS rows,
       SUM(CASE WHEN d.{C['d_bal']} = 0 THEN 1 ELSE 0 END) AS zero_bal_rows,
       SUM(CASE WHEN d.{C['d_status']} = 'C' THEN 1 ELSE 0 END) AS status_c_rows,
       ROUND(PERCENTILE_APPROX(ABS(d.{C['d_bal']}), 0.5), 2) AS median_abs_bal
FROM   dep d
JOIN   acct_span s ON s.acct = d.{C['d_acct']}
WHERE  s.closed_dt IS NOT NULL AND d.{C['d_load']} > s.closed_dt
GROUP  BY 1 ORDER BY 1
"""))

# Q2: silent disappearance — gone from the panel, no closed_dt, no 'C'
show("Silent disappearances (last_seen well before panel end, no closure evidence)", q(f"""
SELECT DATE_FORMAT(last_seen, 'yyyy-MM') AS last_seen_month,
       COUNT(*)                          AS accts
FROM   acct_span
WHERE  closed_dt IS NULL
  AND  ever_status_c = 0
  AND  last_seen < DATE_SUB('{END}', 31)
GROUP  BY 1 ORDER BY 1
"""))

# Q3: is any pre-window closure represented at all?
show("Accounts whose closed_dt precedes the window", q(f"""
SELECT COUNT(*) AS accts_closed_before_window
FROM   acct_span
WHERE  closed_dt IS NOT NULL AND closed_dt < '{START}'
"""))

# Left-censoring shape: a spike of first_seen at the window start is expected;
# a spike anywhere else is a scope change worth understanding.
show("First-seen month (panel entry)", q("""
SELECT DATE_FORMAT(first_seen, 'yyyy-MM') AS first_seen_month,
       COUNT(*)                           AS accts,
       SUM(CASE WHEN opened_dt IS NOT NULL
                 AND DATE_FORMAT(opened_dt,'yyyy-MM') = DATE_FORMAT(first_seen,'yyyy-MM')
            THEN 1 ELSE 0 END)            AS opened_same_month
FROM   acct_span GROUP BY 1 ORDER BY 1
"""))

### 1.4 What is `avg_monthly_bal_1`?

Two candidate meanings, distinguishable without asking anyone:

- **Month-to-date running mean** → the value changes across load dates inside a
  month, and at month end equals the mean of that month's daily `balance`.
- **Prior complete month** → the value is constant inside a month and equals the
  mean of the *previous* month's daily `balance`.

Test both against the daily series on a sample of accounts with full history.
If neither matches, print the residuals rather than guessing.

In [ ]:
sample_accts = spark.sql(f"""                                       -- noqa: F821
SELECT acct FROM acct_span
WHERE  n_days > 400 AND closed_dt IS NULL
ORDER  BY RAND(42) LIMIT {SEM_SAMPLE_ACCTS}
""")
sample_accts.createOrReplaceTempView("sample_accts")

daily = spark.sql(f"""                                              -- noqa: F821
SELECT d.{C['d_acct']}                          AS acct,
       DATE_FORMAT(d.{C['d_load']}, 'yyyy-MM')  AS ym,
       d.{C['d_load']}                          AS load_dt,
       CAST(d.{C['d_bal']}  AS DOUBLE)          AS bal,
       CAST(d.{C['d_avg1']} AS DOUBLE)          AS avg1
FROM   dep d JOIN sample_accts s ON s.acct = d.{C['d_acct']}
""")
daily.createOrReplaceTempView("daily")

# (a) Does avg1 vary within an account-month?
show("Distinct avg_monthly_bal_1 values within an account-month", q("""
SELECT n_distinct_avg1, COUNT(*) AS acct_months
FROM ( SELECT acct, ym, COUNT(DISTINCT avg1) AS n_distinct_avg1
       FROM daily GROUP BY acct, ym )
GROUP BY 1 ORDER BY 1 LIMIT 30
"""))

# (b) Head-to-head against the two candidate definitions, at month end.
sem = q("""
WITH m AS (
  SELECT acct, ym,
         AVG(bal)                                          AS mean_bal_this_month,
         MAX(CASE WHEN load_dt = mx_dt THEN avg1 END)      AS avg1_at_month_end
  FROM ( SELECT d.*, MAX(load_dt) OVER (PARTITION BY acct, ym) AS mx_dt FROM daily d )
  GROUP BY acct, ym
), lagged AS (
  SELECT acct, ym, mean_bal_this_month, avg1_at_month_end,
         LAG(mean_bal_this_month) OVER (PARTITION BY acct ORDER BY ym) AS mean_bal_prior_month
  FROM m
)
SELECT
  COUNT(*) AS acct_months,
  ROUND(PERCENTILE_APPROX(ABS(avg1_at_month_end - mean_bal_this_month)
        / NULLIF(ABS(mean_bal_this_month), 0), 0.5), 6) AS med_relerr_vs_THIS_month,
  ROUND(PERCENTILE_APPROX(ABS(avg1_at_month_end - mean_bal_prior_month)
        / NULLIF(ABS(mean_bal_prior_month), 0), 0.5), 6) AS med_relerr_vs_PRIOR_month,
  SUM(CASE WHEN ABS(avg1_at_month_end - mean_bal_this_month)  <= 0.01 THEN 1 ELSE 0 END) AS exact_this,
  SUM(CASE WHEN ABS(avg1_at_month_end - mean_bal_prior_month) <= 0.01 THEN 1 ELSE 0 END) AS exact_prior
FROM lagged WHERE mean_bal_prior_month IS NOT NULL
""")
show("avg_monthly_bal_1 semantics — the lower median relative error wins", sem)
print("\nRead: near-zero med_relerr_vs_THIS_month  -> month-to-date / current-month average."
      "\n      near-zero med_relerr_vs_PRIOR_month -> lag-1 complete month."
      "\n      Neither near zero -> it is neither; inspect a single account below.")

show("One account, one year, side by side", q("""
WITH one AS (SELECT acct FROM sample_accts LIMIT 1)
SELECT d.ym, MAX(d.load_dt) AS month_end, ROUND(AVG(d.bal),2) AS mean_bal_this_month,
       ROUND(MAX(d.avg1),2) AS max_avg1, ROUND(MIN(d.avg1),2) AS min_avg1
FROM daily d JOIN one o ON o.acct = d.acct
GROUP BY d.ym ORDER BY d.ym LIMIT 18
"""))

### 1.5 Balance distribution

Negatives are expected (overdrafts). What matters for the label is how common
a persistently zero or near-zero balance is, since that is one candidate
definition of an empty account.

In [ ]:
show("Balance distribution on the last load date", q(f"""
WITH latest AS (SELECT MAX({C['d_load']}) AS mx FROM dep)
SELECT COUNT(*) AS accts,
       SUM(CASE WHEN {C['d_bal']} <  0 THEN 1 ELSE 0 END) AS negative,
       SUM(CASE WHEN {C['d_bal']} =  0 THEN 1 ELSE 0 END) AS exactly_zero,
       SUM(CASE WHEN {C['d_bal']} > 0 AND {C['d_bal']} <    100 THEN 1 ELSE 0 END) AS under_100,
       SUM(CASE WHEN {C['d_bal']} >= 100 AND {C['d_bal']} < 25000 THEN 1 ELSE 0 END) AS b_100_25k,
       SUM(CASE WHEN {C['d_bal']} >= 25000 THEN 1 ELSE 0 END) AS over_25k,
       ROUND(PERCENTILE_APPROX({C['d_bal']}, ARRAY(0.01,0.1,0.5,0.9,0.99)), 0) AS pctiles
FROM   dep d CROSS JOIN latest l WHERE d.{C['d_load']} = l.mx
"""))

### 1.6 Closure and emptiness — four definitions, and how much they agree

You asked for the flag *and* alternative definitions, plus their relationship.
Four candidates, all at account level:

| | Definition |
|---|---|
| **A** | `acct_status = 'C'` on the last row seen |
| **B** | `closed_dt` is populated |
| **C** | balance below `EMPTY_FLOOR` for 3 consecutive month-ends, and stays there |
| **D** | account stops appearing in the panel before the window ends |

The cross-tab is the deliverable. If A and B agree almost perfectly, the label
is easy. Where they disagree, the disagreement is the interesting population:
an account with a `closed_dt` but no `C` status is a pending closure; a `C`
status with no date is a back-dated one.

In [ ]:
EMPTY_FLOOR = 100.0

month_end = spark.sql(f"""                                          -- noqa: F821
WITH me AS (
  SELECT d.*, DATE_FORMAT(d.{C['d_load']}, 'yyyy-MM') AS ym,
         ROW_NUMBER() OVER (PARTITION BY d.{C['d_acct']},
                            DATE_FORMAT(d.{C['d_load']}, 'yyyy-MM')
                            ORDER BY d.{C['d_load']} DESC) AS rn
  FROM dep d
)
SELECT {C['d_acct']} AS acct, ym, {C['d_load']} AS load_dt,
       CAST({C['d_bal']} AS DOUBLE) AS bal, {C['d_status']} AS status,
       {C['d_cust']} AS cust, {C['d_fam']} AS fam, {C['d_close']} AS closed_dt
FROM   me WHERE rn = 1
""")
month_end.createOrReplaceTempView("month_end")
month_end.cache().count()

show("Closure definitions — cross-tab", q(f"""
WITH lastrow AS (
  SELECT acct, ym, status, bal, closed_dt,
         ROW_NUMBER() OVER (PARTITION BY acct ORDER BY ym DESC) AS rn
  FROM month_end
), empty3 AS (
  SELECT acct, MAX(CASE WHEN run3 = 3 THEN 1 ELSE 0 END) AS ever_empty_3m
  FROM ( SELECT acct, ym,
                SUM(CASE WHEN bal < {EMPTY_FLOOR} THEN 1 ELSE 0 END)
                  OVER (PARTITION BY acct ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS run3
         FROM month_end )
  GROUP BY acct
)
SELECT CASE WHEN l.status = 'C' THEN 1 ELSE 0 END      AS A_status_c,
       CASE WHEN l.closed_dt IS NOT NULL THEN 1 ELSE 0 END AS B_closed_dt,
       COALESCE(e.ever_empty_3m, 0)                    AS C_empty_3m,
       CASE WHEN s.last_seen < DATE_SUB('{END}', 31) THEN 1 ELSE 0 END AS D_vanished,
       COUNT(*)                                        AS accts
FROM   lastrow l
JOIN   acct_span s ON s.acct = l.acct
LEFT   JOIN empty3 e ON e.acct = l.acct
WHERE  l.rn = 1
GROUP  BY 1, 2, 3, 4
ORDER  BY accts DESC
"""))

show("Closures per month — the observable departure ceiling", q("""
SELECT DATE_FORMAT(closed_dt, 'yyyy-MM') AS closed_month, COUNT(*) AS accts
FROM   acct_span WHERE closed_dt IS NOT NULL
GROUP  BY 1 ORDER BY 1
"""))

---
## 2. Payments staging table

### 2.1 Null-pattern topology

The stated rule is that the two `mdm_id` columns encode direction. This
enumerates every combination that actually occurs, with volume and dollars, so
the direction logic is written against the observed shape rather than the
intended one. Combinations to look for specifically: both `mdm_id`s null; both
non-null *and* a counterparty id present; an `mdm_id` present with a null
deposit account.

In [ ]:
show("Null topology across the id columns", q(f"""
SELECT CASE WHEN {C['mdm_p']}  IS NULL THEN 0 ELSE 1 END AS mdm_pays,
       CASE WHEN {C['mdm_r']}  IS NULL THEN 0 ELSE 1 END AS mdm_recv,
       CASE WHEN {C['acct_p']} IS NULL THEN 0 ELSE 1 END AS acct_pays,
       CASE WHEN {C['acct_r']} IS NULL THEN 0 ELSE 1 END AS acct_recv,
       CASE WHEN {C['cpty_id']} IS NULL THEN 0 ELSE 1 END AS cpty_id,
       CASE WHEN {C['cpty_nm']} IS NULL THEN 0 ELSE 1 END AS cpty_nm,
       COUNT(*)                            AS rows,
       ROUND(SUM({C['amt']}) / 1e6, 1)     AS amount_mm
FROM   pay
GROUP  BY 1,2,3,4,5,6
ORDER  BY rows DESC
"""))

show("mdm_id present but deposit account null — by rail", q(f"""
SELECT {C['rail']} AS rail, {C['cat']} AS category,
       SUM(CASE WHEN {C['mdm_p']} IS NOT NULL AND {C['acct_p']} IS NULL THEN 1 ELSE 0 END) AS pays_missing_acct,
       SUM(CASE WHEN {C['mdm_r']} IS NOT NULL AND {C['acct_r']} IS NULL THEN 1 ELSE 0 END) AS recv_missing_acct,
       COUNT(*) AS rows
FROM   pay GROUP BY 1,2 ORDER BY rows DESC
"""))

### 2.2 One economic payment — one row, or two?

`transaction_id` being unique per row does not settle this. Two failure modes
survive a unique id:

1. An internal payment written **twice**, once from each side, each row
   carrying its own id and only one `mdm_id` populated.
2. An internal payment written twice with **both** sides populated on each row.

Both are tested against a chance baseline. Date-plus-amount collisions happen
naturally, so the raw pair count means nothing without knowing how often two
unrelated payments share a date and an amount.

In [ ]:
show("transaction_id uniqueness", q(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT {C['txn_id']}) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT {C['txn_id']}) AS repeated_ids
FROM   pay
"""))

# Content-key duplicates: same date, amount, rail and both endpoints.
show("Content-key duplicate groups (different transaction_ids, identical content)", q(f"""
WITH keyed AS (
  SELECT {C['txn_dt']} AS dt, {C['amt']} AS amt, {C['rail']} AS rail,
         COALESCE({C['acct_p']}, '~') AS ap, COALESCE({C['acct_r']}, '~') AS ar,
         COALESCE({C['cpty_id']}, '~') AS cp,
         COALESCE({C['mdm_p']}, '~')  AS mp, COALESCE({C['mdm_r']}, '~') AS mr,
         {C['txn_id']} AS txn
  FROM pay
)
SELECT n_rows_in_group, COUNT(*) AS n_groups, SUM(n_rows_in_group) AS rows_involved
FROM ( SELECT dt, amt, rail, ap, ar, cp, mp, mr, COUNT(*) AS n_rows_in_group
       FROM keyed GROUP BY dt, amt, rail, ap, ar, cp, mp, mr )
GROUP BY 1 ORDER BY 1
"""))

show("Examples of content-key duplicate groups", q(f"""
WITH g AS (
  SELECT {C['txn_dt']} AS dt, {C['amt']} AS amt, {C['rail']} AS rail,
         COALESCE({C['acct_p']},'~') AS ap, COALESCE({C['acct_r']},'~') AS ar,
         COALESCE({C['cpty_id']},'~') AS cp, COUNT(*) AS n
  FROM pay GROUP BY 1,2,3,4,5,6 HAVING COUNT(*) > 1
)
SELECT p.* FROM pay p JOIN g
  ON  p.{C['txn_dt']} = g.dt AND p.{C['amt']} = g.amt AND p.{C['rail']} = g.rail
  AND COALESCE(p.{C['acct_p']},'~') = g.ap
  AND COALESCE(p.{C['acct_r']},'~') = g.ar
  AND COALESCE(p.{C['cpty_id']},'~') = g.cp
ORDER BY p.{C['txn_dt']}, p.{C['amt']}
""", n=40))

# Split-leg test: is a C2C payment written as two one-sided rows?
# Restrict to a short window so the self-join stays bounded, and report the
# chance baseline alongside the match count.
show("Split-leg test — one-sided rows that pair up on date+amount", q(f"""
WITH w AS (
  SELECT * FROM pay
  WHERE {C['txn_dt']} >= DATE_SUB('{END}', {DUP_SAMPLE_DAYS}) AND {C['txn_dt']} <= '{END}'
),
out_only AS (   -- PNC pays, receiver looks external
  SELECT {C['txn_dt']} AS dt, {C['amt']} AS amt, {C['acct_p']} AS acct, {C['txn_id']} AS txn
  FROM w WHERE {C['mdm_p']} IS NOT NULL AND {C['mdm_r']} IS NULL
),
in_only AS (    -- PNC receives, payer looks external
  SELECT {C['txn_dt']} AS dt, {C['amt']} AS amt, {C['acct_r']} AS acct, {C['txn_id']} AS txn
  FROM w WHERE {C['mdm_r']} IS NOT NULL AND {C['mdm_p']} IS NULL
),
both AS (       -- genuine two-sided rows, for comparison
  SELECT COUNT(*) AS n FROM w WHERE {C['mdm_p']} IS NOT NULL AND {C['mdm_r']} IS NOT NULL
)
SELECT (SELECT COUNT(*) FROM out_only)                         AS out_only_rows,
       (SELECT COUNT(*) FROM in_only)                          AS in_only_rows,
       (SELECT n FROM both)                                    AS two_sided_rows,
       COUNT(*)                                                AS paired_on_dt_amt,
       COUNT(DISTINCT o.txn)                                   AS distinct_out_matched
FROM   out_only o JOIN in_only i ON o.dt = i.dt AND o.amt = i.amt
"""))
print("\nRead: if paired_on_dt_amt is a large share of out_only_rows AND two_sided_rows is "
      "near zero, internal payments are being written as two one-sided rows and every "
      "C2C dollar is double counted. Compare against the amount-collision baseline below.")

show("Chance baseline — how often do unrelated payments share date and amount", q(f"""
SELECT ROUND(AVG(n), 3) AS mean_rows_per_dt_amt,
       ROUND(PERCENTILE_APPROX(n, 0.99), 1) AS p99_rows_per_dt_amt,
       SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS dt_amt_keys_with_collision,
       COUNT(*) AS dt_amt_keys
FROM ( SELECT {C['txn_dt']} AS dt, {C['amt']} AS amt, COUNT(*) AS n
       FROM pay
       WHERE {C['txn_dt']} >= DATE_SUB('{END}', {DUP_SAMPLE_DAYS}) AND {C['txn_dt']} <= '{END}'
       GROUP BY 1, 2 )
"""))

# Does a two-sided row ever also carry a counterparty id? It should not.
show("Two-sided rows carrying a counterparty id", q(f"""
SELECT COUNT(*) AS rows FROM pay
WHERE  {C['mdm_p']} IS NOT NULL AND {C['mdm_r']} IS NOT NULL AND {C['cpty_id']} IS NOT NULL
"""))

### 2.3 Rail and category

`category` is where on-us / off-us lives. A book transfer between two PNC
accounts is a different event from a payment leaving the franchise, and the
attrition metrics need to tell them apart.

In [ ]:
show("rail x category", q(f"""
SELECT {C['rail']} AS rail, {C['cat']} AS category,
       COUNT(*) AS rows, ROUND(SUM({C['amt']}) / 1e6, 1) AS amount_mm,
       SUM(CASE WHEN {C['mdm_p']} IS NOT NULL AND {C['mdm_r']} IS NOT NULL THEN 1 ELSE 0 END) AS c2c_rows,
       SUM(CASE WHEN {C['cpty_id']} IS NOT NULL THEN 1 ELSE 0 END) AS cpty_rows
FROM   pay GROUP BY 1, 2 ORDER BY rows DESC
"""))

show("Monthly volume by rail — coverage over the window", q(f"""
SELECT DATE_FORMAT({C['txn_dt']}, 'yyyy-MM') AS ym, {C['rail']} AS rail,
       COUNT(*) AS rows, ROUND(SUM({C['amt']}) / 1e6, 1) AS amount_mm
FROM   pay GROUP BY 1, 2 ORDER BY 1, 2
"""))

### 2.4 Amount

In [ ]:
show("Amount distribution", q(f"""
SELECT COUNT(*) AS rows,
       SUM(CASE WHEN {C['amt']} IS NULL THEN 1 ELSE 0 END) AS null_amt,
       SUM(CASE WHEN {C['amt']} <  0 THEN 1 ELSE 0 END) AS negative,
       SUM(CASE WHEN {C['amt']} =  0 THEN 1 ELSE 0 END) AS zero,
       ROUND(PERCENTILE_APPROX({C['amt']}, ARRAY(0.01,0.5,0.9,0.99,0.999)), 2) AS pctiles,
       ROUND(MAX({C['amt']}), 2) AS max_amt
FROM   pay
"""))

### 2.5 Counterparty fields

`cpty_name` is what makes `same_name_outflow_flag` possible, so its coverage
is the gate on the strongest signal in Group A. Two numbers matter: what share
of outflow *value* carries a name, and how many outflows exact-match the
client's own name.

Exact match is reported two ways. Raw exact, and exact after case-folding and
stripping punctuation and whitespace. That second one is not entity
resolution; it is the difference between `ACME LLC` and `Acme  LLC.`, and the
gap between the two counts tells you whether the raw match is leaving signal
on the table.

In [ ]:
show("cpty_name coverage by direction", q(f"""
SELECT CASE WHEN {C['mdm_p']} IS NOT NULL THEN 'pnc_outbound' ELSE 'pnc_inbound' END AS direction,
       {C['rail']} AS rail,
       COUNT(*) AS rows,
       SUM(CASE WHEN {C['cpty_nm']} IS NULL OR TRIM({C['cpty_nm']}) = '' THEN 1 ELSE 0 END) AS name_missing,
       ROUND(SUM(CASE WHEN {C['cpty_nm']} IS NOT NULL AND TRIM({C['cpty_nm']}) <> ''
                      THEN {C['amt']} ELSE 0 END) / NULLIF(SUM({C['amt']}), 0), 4) AS amt_share_named
FROM   pay WHERE {C['cpty_id']} IS NOT NULL
GROUP  BY 1, 2 ORDER BY rows DESC
"""))

show("same_name outflow — raw exact vs normalised exact", q(f"""
WITH o AS (
  SELECT {C['nm_p']} AS ego_nm, {C['cpty_nm']} AS cp_nm, {C['amt']} AS amt,
         REGEXP_REPLACE(UPPER(TRIM({C['nm_p']})),   '[^A-Z0-9]', '') AS ego_k,
         REGEXP_REPLACE(UPPER(TRIM({C['cpty_nm']})),'[^A-Z0-9]', '') AS cp_k
  FROM pay
  WHERE {C['mdm_p']} IS NOT NULL AND {C['cpty_id']} IS NOT NULL
    AND {C['cpty_nm']} IS NOT NULL AND TRIM({C['cpty_nm']}) <> ''
)
SELECT COUNT(*) AS named_outflow_rows,
       SUM(CASE WHEN ego_nm = cp_nm THEN 1 ELSE 0 END)                      AS exact_raw,
       SUM(CASE WHEN ego_k  = cp_k AND ego_k <> '' THEN 1 ELSE 0 END)       AS exact_normalised,
       ROUND(SUM(CASE WHEN ego_k = cp_k AND ego_k <> '' THEN amt ELSE 0 END) / 1e6, 1) AS matched_amount_mm,
       COUNT(DISTINCT CASE WHEN ego_k = cp_k AND ego_k <> '' THEN ego_k END) AS distinct_clients_matched
FROM   o
"""))

show("unq_cpty_acct_id — format and stability", q(f"""
SELECT COUNT(DISTINCT {C['cpty_id']}) AS distinct_cpty_accts,
       SUM(CASE WHEN {C['cpty_id']} NOT RLIKE '^[0-9]{{9}}-' THEN 1 ELSE 0 END) AS not_rtn_dash_prefixed,
       COUNT(DISTINCT SUBSTRING_INDEX({C['cpty_id']}, '-', 1)) AS distinct_rtns
FROM   pay WHERE {C['cpty_id']} IS NOT NULL
"""))

show("Name and bank variants per counterparty account (top offenders)", q(f"""
SELECT n_names, COUNT(*) AS cpty_accts FROM (
  SELECT {C['cpty_id']} AS cp, COUNT(DISTINCT {C['cpty_nm']}) AS n_names
  FROM pay WHERE {C['cpty_id']} IS NOT NULL AND {C['cpty_nm']} IS NOT NULL
  GROUP BY 1 )
GROUP BY 1 ORDER BY 1 LIMIT 20
"""))

show("FI-name variants per routing number", q(f"""
SELECT n_fi_names, COUNT(*) AS rtns FROM (
  SELECT SUBSTRING_INDEX({C['cpty_id']}, '-', 1) AS rtn,
         COUNT(DISTINCT {C['cpty_fi']}) AS n_fi_names
  FROM pay WHERE {C['cpty_id']} IS NOT NULL AND {C['cpty_fi']} IS NOT NULL
  GROUP BY 1 )
GROUP BY 1 ORDER BY 1 LIMIT 20
"""))

---
## 3. The join

### 3.1 Account match rate

Formats agree, so this is a coverage measure rather than a format check: how
much of the payment table belongs to an account we hold deposit data for, and
how much of the deposit book shows any payment activity at all. The second
number sizes the study population; the first sizes what we throw away.

In [ ]:
spark.sql("""                                                       -- noqa: F821
CREATE OR REPLACE TEMPORARY VIEW pay_legs AS
SELECT txn_id, dt, amt, rail, category, direction, acct, mdm, cpty_id, cpty_nm, cpty_fi, other_mdm
FROM (
  SELECT {t} AS txn_id, {d} AS dt, {a} AS amt, {r} AS rail, {c} AS category,
         'out' AS direction, {ap} AS acct, {mp} AS mdm, {ci} AS cpty_id,
         {cn} AS cpty_nm, {cf} AS cpty_fi, {mr} AS other_mdm
  FROM pay WHERE {mp} IS NOT NULL
  UNION ALL
  SELECT {t}, {d}, {a}, {r}, {c},
         'in', {ar}, {mr}, {ci}, {cn}, {cf}, {mp}
  FROM pay WHERE {mr} IS NOT NULL
)
""".format(t=C['txn_id'], d=C['txn_dt'], a=C['amt'], r=C['rail'], c=C['cat'],
           ap=C['acct_p'], ar=C['acct_r'], mp=C['mdm_p'], mr=C['mdm_r'],
           ci=C['cpty_id'], cn=C['cpty_nm'], cf=C['cpty_fi']))

show("Leg-level join coverage against the deposit account universe", q("""
WITH u AS (SELECT DISTINCT acct FROM acct_span)
SELECT l.direction,
       COUNT(*)                                              AS legs,
       SUM(CASE WHEN l.acct IS NULL THEN 1 ELSE 0 END)       AS leg_acct_null,
       SUM(CASE WHEN u.acct IS NOT NULL THEN 1 ELSE 0 END)   AS legs_matched,
       ROUND(SUM(CASE WHEN u.acct IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 4) AS match_rate_rows,
       ROUND(SUM(CASE WHEN u.acct IS NOT NULL THEN l.amt ELSE 0 END)
             / NULLIF(SUM(l.amt), 0), 4)                     AS match_rate_amount
FROM   pay_legs l LEFT JOIN u ON u.acct = l.acct
GROUP  BY l.direction
"""))

show("Deposit accounts with any payment activity, by family", q("""
WITH act AS (SELECT DISTINCT acct FROM pay_legs WHERE acct IS NOT NULL)
SELECT m.fam AS deposit_family,
       COUNT(DISTINCT m.acct) AS accts,
       COUNT(DISTINCT CASE WHEN a.acct IS NOT NULL THEN m.acct END) AS accts_with_payments,
       ROUND(COUNT(DISTINCT CASE WHEN a.acct IS NOT NULL THEN m.acct END)
             / COUNT(DISTINCT m.acct), 4) AS coverage
FROM   month_end m LEFT JOIN act a ON a.acct = m.acct
GROUP  BY m.fam ORDER BY accts DESC
"""))

### 3.2 `mdm_id` ↔ `cust_pwr_id` cardinality

Stated as one-to-one. Worth a cheap confirmation, because the customer-level
roll-up in the next phase depends on it and a violation is silent.

In [ ]:
show("mdm_id to cust_pwr_id cardinality", q(f"""
WITH xw AS (
  SELECT DISTINCT l.mdm AS mdm_id, d.{C['d_cust']} AS cust_pwr_id
  FROM   pay_legs l
  JOIN   ( SELECT DISTINCT {C['d_acct']} AS acct, {C['d_cust']} AS {C['d_cust']} FROM dep ) d
         ON d.acct = l.acct
  WHERE  l.mdm IS NOT NULL
)
SELECT 'cust_pwr_id per mdm_id' AS direction, n, COUNT(*) AS entities
FROM   ( SELECT mdm_id, COUNT(DISTINCT cust_pwr_id) AS n FROM xw GROUP BY 1 )
GROUP  BY 1, 2
UNION ALL
SELECT 'mdm_id per cust_pwr_id', n, COUNT(*)
FROM   ( SELECT cust_pwr_id, COUNT(DISTINCT mdm_id) AS n FROM xw GROUP BY 1 )
GROUP  BY 1, 2
ORDER  BY 1, 2
"""))

### 3.3 Panel shape after scoping

The number that sizes everything downstream: accounts that have both a usable
deposit series and payment activity, and how many of those close inside the
window. Twelve months of the window go to baseline burn-in, so the scorable
span is what is left after that.

In [ ]:
show("Study population sizing", q(f"""
WITH act AS (SELECT DISTINCT acct FROM pay_legs WHERE acct IS NOT NULL),
     s AS (
       SELECT sp.acct, sp.n_days, sp.closed_dt,
              CASE WHEN a.acct IS NOT NULL THEN 1 ELSE 0 END AS has_payments
       FROM   acct_span sp LEFT JOIN act a ON a.acct = sp.acct
     )
SELECT COUNT(*)                                                          AS accts_total,
       SUM(has_payments)                                                 AS with_payments,
       SUM(CASE WHEN n_days >= 250 THEN 1 ELSE 0 END)                    AS with_12m_history,
       SUM(CASE WHEN n_days >= 250 AND has_payments = 1 THEN 1 ELSE 0 END) AS both,
       SUM(CASE WHEN closed_dt IS NOT NULL AND has_payments = 1 THEN 1 ELSE 0 END) AS closures_with_payments,
       SUM(CASE WHEN closed_dt >= '2025-01-01' AND has_payments = 1 THEN 1 ELSE 0 END) AS closures_after_burnin
FROM   s
"""))

---
## 4. Open items to carry forward

Fill these in from the output above before the extraction notebook is written.

| # | Item | Where answered | Decision needed |
|---|---|---|---|
| 1 | `deposit_family` values in scope | §1.2 | yours |
| 2 | Account persists after `closed_dt`? | §1.3 | mechanical |
| 3 | Panel survivor-only from window start? | §1.3 | mechanical; caps the study |
| 4 | `avg_monthly_bal_1` meaning | §1.4 | mechanical |
| 5 | Which closure definition is the label | §1.6 | yours, informed by the cross-tab |
| 6 | One row or two per economic payment | §2.2 | mechanical; blocks all dollar totals |
| 7 | Book-transfer / on-us category handling | §2.3 | yours |
| 8 | `cpty_name` coverage sufficient for Group A? | §2.5 | mechanical, then yours |
| 9 | Legs dropped for want of a deposit account | §3.1 | mechanical |
| 10 | Scorable months after 12-month burn-in | §3.3 | mechanical |

**Note on the window.** Payments and deposits both start 2024-01, and the brief
requires twelve months of deposit baseline before a T0 can be dated. That puts
the first scorable month at 2025-01 and leaves roughly nineteen months of
scorable client-months to 2026-07, before any survivorship loss from §1.3.

In [ ]:
for v in ("acct_span", "month_end"):
    try:
        spark.catalog.uncacheTable(v)                               # noqa: F821
    except Exception:
        pass
print("done")